In [5]:
import pandas as pd
import numpy as np
import glob
import os

In [12]:
!rm -rf Portofolio
!git clone https://github.com/dewitrilestari/Portofolio.git

path_folder = os.path.join("Portofolio", "Forecasting", "Raw Data")
semua_file = glob.glob(os.path.join(path_folder, "*.xlsx"))
semua_file.sort()

list_df = []
for file in semua_file:
    try:
        df_temp = pd.read_excel(file, engine='openpyxl', header=7)
        df_temp.columns = df_temp.columns.astype(str).str.strip()
        df_temp = df_temp.loc[:, ~df_temp.columns.str.contains('^Unnamed|^nan', case=False)]

        for col in df_temp.columns:
            if 'tang' in col.lower() or 'date' in col.lower():
                df_temp = df_temp.rename(columns={col: 'Tanggal'})
                break

        df_temp = df_temp.dropna(subset=['Tanggal'])
        list_df.append(df_temp)
    except Exception as e:
        pass

if len(list_df) > 0:
    df_gabung = pd.concat(list_df, ignore_index=True)

df_gabung['Tanggal'] = pd.to_datetime(df_gabung['Tanggal'], dayfirst=True, errors='coerce')
df_gabung = df_gabung.dropna(subset=['Tanggal'])
df_gabung = df_gabung.sort_values('Tanggal').reset_index(drop=True)

df_gabung = df_gabung.replace([9999, 9999.0, '9999', '9999.0'], np.nan)

if 'RR' in df_gabung.columns:
    df_gabung['RR'] = df_gabung['RR'].replace(['8888', '8888.0', 8888, 8888.0], 0)
    df_gabung['RR'] = pd.to_numeric(df_gabung['RR'], errors='coerce').fillna(0)

for col in df_gabung.columns:
    if col not in ['Tanggal', 'DDD_CAR']:
        df_gabung[col] = pd.to_numeric(df_gabung[col], errors='coerce')
        df_gabung[col] = df_gabung[col].interpolate(method='linear')

if 'DDD_CAR' in df_gabung.columns:
    df_gabung['DDD_CAR'] = df_gabung['DDD_CAR'].ffill().bfill()

display(df_gabung.head(10))

Cloning into 'Portofolio'...
remote: Enumerating objects: 204, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (195/195), done.
remote: Total 204 (delta 77), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (204/204), 4.11 MiB | 14.81 MiB/s, done.
Resolving deltas: 100% (77/77), done.


,Tanggal,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,FF_AVG,DDD_CAR
0,2024-07-14,18.8,30.7,24.1,74.0,0.0,7.5,4.0,130.0,2.0,W
1,2024-07-15,19.8,30.5,24.3,74.0,0.0,8.0,6.0,180.0,2.0,S
2,2024-07-16,19.6,30.5,24.6,77.0,0.0,8.0,5.0,280.0,2.0,W
3,2024-07-17,21.6,30.6,25.7,80.0,0.0,8.0,4.0,260.0,2.0,W
4,2024-07-18,21.3,31.7,25.6,79.0,0.0,8.0,3.0,210.0,2.0,W
5,2024-07-19,20.4,32.2,26.1,72.0,0.0,5.9,5.0,220.0,2.0,S
6,2024-07-20,19.6,30.6,24.3,76.0,0.0,8.0,5.0,300.0,2.0,W
7,2024-07-21,19.6,31.5,24.6,75.0,0.0,8.0,5.0,260.0,3.0,W
8,2024-07-22,20.5,30.4,24.5,81.0,0.0,8.0,4.0,270.0,2.0,SW
9,2024-07-23,21.7,30.7,25.4,80.0,0.0,8.0,6.0,270.0,3.0,W


In [13]:
df_gabung.to_excel('Juli 2024 - Juli 2026.xlsx', index=False)